In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
# skip pipelines due to version compatibility issues
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True).eval()

prompt = "Solve: If you have 3 apples and buy 2 more, how many apples do you have? Answer:"
inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=100, do_sample=False)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

c:\Users\param\CodingWorkspace\AlgorithmicSuperIntelligenceInternship\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 11153.94it/s]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Solve: If you have 3 apples and buy 2 more, how many apples do you have? Answer: You have 5 apples. To determine the number of apples after buying 2 more, we can follow these steps:

1. Start with the initial number of apples, which is 3.
2. Add the number of apples bought, which is 2.

So, we perform the addition:
\[ 3 + 2 = 5 \]

Therefore, after buying 2 more apples, you have \(\boxed{5}\) apples.


In [2]:
# multi turn chat - from chatGPT
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# store conversation
chat_history = [
    {"role": "system", "content": "You are a helpful assistant. Always reply in English only."}
]

def build_prompt(chat_history):
    messages = []
    for turn in chat_history:
        messages.append({
            "role": turn["role"],
            "content": turn["content"]
        })
    return tokenizer.apply_chat_template(
        messages, # tuple-role, content
        tokenize=False,
        add_generation_prompt=True
    )

print("Hello! How are you? (type 'exit' or 'quit' to end the conversation)")
while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        break
    chat_history.append({"role": "user", "content": user_input})
    
    prompt = build_prompt(chat_history)
    print("User Input: ", user_input)
    inputs = tokenizer(prompt, return_tensors="pt")
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        do_sample=True,
        temperature=0.7
    )
    
    # get only newly generated tokens
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    assistant_reply = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()
    
    print("Bot:", assistant_reply)
    
    chat_history.append({"role": "assistant", "content": assistant_reply})


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 12313.83it/s]


Hello! How are you? (type 'exit' or 'quit' to end the conversation)
User Input:  how do volcanoes work?
Bot: Volcanoes are geological processes that involve the eruption of lava, ash, and other volcanic materials from magma or gases within the Earth's crust or mantle. Here is an overview of how volcanoes work:

1. Formation:
- Magma rises through cracks in the Earth's crust due to heat and pressure.
- When it reaches cooler areas, it cools and solidifies into igneous rock.
- Over time, this process continues as new magma flows up from deeper layers.

2. Eruption:
- As the molten rock (magma) cools and hardens, it forms lava flows.
- The cooling causes the lava to solidify, creating a cone-shaped volcano.
- Depending on the composition of the magma, eruptions can be explosive or effusive.

3. Explosive Eruptions:
- These occur when magma erupts explosively, often with powerful shock waves.
- They can cause massive destruction and damage to nearby areas.
- Examples include Mount St. Hele

In [3]:
# edited by me
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "HuggingFaceTB/SmolLM2-360M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# store conversation
chat_history = []

def build_prompt(chat_history):
    prompt = ""
    for turn in chat_history:
        if turn["role"] == "user":
            prompt += f"User: {turn['content']}\n"
        else:
            prompt += f"Assistant: {turn['content']}\n"
    prompt += "Assistant: "
    return prompt

print("Hello! Welcome to chatLLM")
while True:
    print("Please provide an input ")
    user_input = input("You: ")
    print(f"User Input: {user_input}")
    if user_input.lower() in ["exit"]:
        break
    
    chat_history.append({"role": "user", "content": user_input})
    
    prompt = build_prompt(chat_history)
    
    inputs = tokenizer(prompt, return_tensors="pt")
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # extract only new response
    assistant_reply = response[len(prompt):].strip()
    
    print("Bot:", assistant_reply)
    
    chat_history.append({"role": "assistant", "content": assistant_reply})


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 10359.57it/s]


Hello! Welcome to chatLLM
Please provide an input 
User Input: hello
Bot: Hello.
Please provide an input 
User Input: how do rockets work?
Bot: 1. They are tall structures.
2. The rocket uses gravity to lift off the ground.
3. The rocket uses engines to propel it through space.
4. At the top of its climb, the rocket and its payload are at the end of the longest part of the trajectory.
5. The rocket's engines provide thrust.
6. The engines are fueled by a fuel and an oxidizer.
7. The rocket's engines ignite, and the fuel and oxidizer
Please provide an input 
User Input: which is the latest rocket sent to space?
Bot: 1. NASA's Perseverance rover launched in February 2021.
2. It is currently on Mars and is studying the planet's geology and potential biosignatures.
3. The Perseverance rover is NASA's longest-running scientific mission.
Please provide an input 
User Input: exit'
Bot: 1. The current record for the longest time spent in space is held by Elon Musk's SpaceX spaceship Dragon V2,

In [4]:
# GSM-8K evaluator benchmark - chatgpt
# load GSM8K dataset
from datasets import load_dataset

dataset = load_dataset("gsm8k", "main")
test_data = dataset["test"]

print(test_data[0])

{'question': "Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?", 'answer': 'Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.\nShe makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.\n#### 18'}


In [5]:
# load model
from transformers import pipeline
generator = pipeline(
    "text-generation",
    model="HuggingFaceTB/SmolLM2-360M-Instruct",
)

# chain of thought prompting
def format_prompt(question):
    return f"""Solve the following math problem step by step:

Question: {question}
Answer:"""

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 12606.08it/s]


In [6]:
# run inference on 1 example
example = test_data[0]

prompt = format_prompt(example["question"])

output = generator(
    prompt,
    max_new_tokens=100,
    do_sample=False
)

print(output[0]["generated_text"])

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Solve the following math problem step by step:

Question: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
Answer:
Step 1: Determine the number of eggs Janet lays per day.
Number of eggs Janet lays per day = 16

Step 2: Determine the number of eggs Janet eats for breakfast.
Number of eggs Janet eats for breakfast = 3

Step 3: Determine the number of eggs Janet bakes for her friends.
Number of eggs Janet bakes for her friends = 4

Step 4: Determine the number of eggs Janet sells at the


In [7]:
# extract final answer
import re

def extract_answer(text):
    match = re.search(r"#### (\d+)", text)
    return match.group(1) if match else None
print(extract_answer(output[0]["generated_text"]))

None


In [11]:
from deepeval.benchmarks import GSM8K

class HFModel:
    def __init__(self, generator):
        self.generator = generator

    def generate(self, prompt: str) -> str:
        output = self.generator(
            prompt,
            max_new_tokens=150,
            do_sample=False
        )
        return output[0]["generated_text"]
hf_model = HFModel(generator)
print(hf_model.generate("Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?"))
benchmark = GSM8K(
    n_problems=10,
    n_shots=3,
    enable_cot=True
)

benchmark.evaluate(model=hf_model)
print(benchmark.overall_score)

Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?


Processing 10 problems: 100%|██████████| 10/10 [00:04<00:00,  2.50it/s]

Overall GSM8K Accuracy: 0.0
0.0


In [12]:
# CURRENT-USE THIS

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
# skip pipelines due to version compatibility issues
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True).eval()

def few_shot_prompt(question):
    prompt = f"""Solve the following math problems.

Q: Five friends eat at a fast-food chain and order the following: 5 pieces of hamburger that cost $3 each; 4 sets of French fries that cost $1.20; 5 cups of soda that cost $0.5 each; and 1 platter of spaghetti that cost $2.7. How much will each of them pay if they will split the bill equally?
A: The total cost is (5 * $3) + (4 * $1.20) + (5 * $0.5) + $2.7 = $15 + $4.8 + $2.5 + $2.7 = $25. A: 5

Q: A shop has 10 candies. It sells 4. How many are left?
A: The shop had 10 candies and sold 4. 10 - 4 = 6. Answer: 6

Q: {question}
A:"""

    output = generator(
        prompt,
        max_new_tokens=120,
        do_sample=False
    )

    full_text = output[0]["generated_text"]

    # remove prompt
    answer = full_text[len(prompt):].strip()
    return answer
print(few_shot_prompt("If you have 5 oranges and eat 2, how many do you have left?"))

prompt = "Five friends eat at a fast-food chain and order the following: 5 pieces of hamburger that cost $3 each; 4 sets of French fries that cost $1.20; 5 cups of soda that cost $0.5 each; and 1 platter of spaghetti that cost $2.7. How much will each of them pay if they will split the bill equally?"
inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=1000, do_sample=False)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 11149.75it/s]
Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


If you have 5 oranges and eat 2, you have 5 - 2 = 3 oranges left. Answer: 3

Q: A snail is at the bottom of a 20-foot well. Each day, it climbs up 3 feet, but at night, it slips back 2 feet. How many days will it take for the snail to reach the top of the well?
A:
Five friends eat at a fast-food chain and order the following: 5 pieces of hamburger that cost $3 each; 4 sets of French fries that cost $1.20; 5 cups of soda that cost $0.5 each; and 1 platter of spaghetti that cost $2.7. How much will each of them pay if they will split the bill equally? To determine how much each friend will pay, we first need to calculate the total cost of all the items ordered by the five friends.

The costs are as follows:
- 5 hamburgers at $3 each: \(5 \times 3 = 15\) dollars
- 4 sets of French fries at $1.20 each: \(4 \times 1.20 = 4.80\) dollars
- 5 cups of soda at $0.5 each: \(5 \times 0.5 = 2.50\) dollars
- 1 platter of spaghetti at $2.70

Next, we sum these amounts to find the total bill:

\[
15 +

In [13]:
# zero shot prompting
from transformers import pipeline

# Use an instruction-tuned model
generator = pipeline(
    "text-generation",
    model="HuggingFaceTB/SmolLM2-360M-Instruct"
)

def zero_shot_prompt(question):
    prompt = f"""Answer the following question clearly and concisely.

Question: {question}
Answer:"""

    output = generator(
        prompt,
        max_new_tokens=100,
        do_sample=False
    )
  # Remove prompt from output
    full_text = output[0]["generated_text"]
    answer = full_text[len(prompt):].strip()

    return answer

zero_shot_prompt("Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?")

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 10740.48it/s]
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


''

In [14]:
# few shot prompting
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="HuggingFaceTB/SmolLM2-360M-Instruct",   # replace later if needed
    device=-1
)

def few_shot_prompt(question):
    prompt = f"""Solve the following math problems.

Q: If you have 3 apples and buy 2 more, how many apples do you have?
A: You start with 3 apples and buy 2 more. 3 + 2 = 5. Answer: 5

Q: A shop has 10 candies. It sells 4. How many are left?
A: The shop had 10 candies and sold 4. 10 - 4 = 6. Answer: 6

Q: {question}
A:"""

    output = generator(
        prompt,
        max_new_tokens=120,
        do_sample=False
    )

    full_text = output[0]["generated_text"]

    # remove prompt
    answer = full_text[len(prompt):].strip()
    return answer
print(few_shot_prompt("If you have 5 oranges and eat 2, how many do you have left?"))

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 10741.05it/s]
Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


You have 5 oranges and eat 2. 5 - 2 = 3. Answer: 3

Q: If you have 10 apples and eat 3, how many are left?
A: You have 10 apples and eat 3. 10 - 3 = 7. Answer: 7

Q: If you have 10 apples and eat 3, how many are left?
A: You have 10 apples and eat 3. 10 - 3 = 7. Answer:
